# Tarefa: Iris - Salvar e carregar o classificador

Após executarmos o processo de tuning da rede neural e coletarmos quais são os melhores parâmetros, é hora de gerar o classificador final para que possamos usar em ambientes comerciais:

- Treinar a rede neural utilizando a base de dados completa utilizando os melhores parâmetros que foram obtidos pelo tuning

- Salvar a estrutura da rede neural

- Carregar a rede neural salva no tópico anterior

- Criar um novo registro (pode ser com dados aleatórios) no mesmo formato dos atributos previsores

- Classificar o novo registro, indicando a que classe ele pertence



## Etapa 1: Importação das bibliotecas

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch
from torch import nn, optim
from torch.nn import functional as F
torch.__version__

'2.8.0+cu128'

## Etapa 2: Base de dados

In [2]:
torch.manual_seed(123)

In [3]:
base = pd.read_csv('iris.csv')
previsores = base.iloc[:, 0:4].values
classe = base.iloc[:, 4].values

In [4]:
encoder = LabelEncoder()
classe = encoder.fit_transform(classe)

In [5]:
previsores = torch.tensor(previsores, dtype = torch.float)
classe = torch.tensor(classe, dtype = torch.long)

In [6]:
dataset = torch.utils.data.TensorDataset(previsores, classe)
train_loader = torch.utils.data.DataLoader(dataset, batch_size = 10, shuffle = True)

## Etapa 3: Construção do modelo

In [7]:
classificador = nn.Sequential(
                nn.Linear(4, 8),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(8, 8),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(8, 3),
                )

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classificador.parameters(), lr = 0.001, weight_decay = 0.0001)

## Etapa 4: Treinamento do modelo

**ciclo de treinamento de uma rede neural**.

Ele executa o processo de aprendizado passo a passo, ajustando os pesos do modelo para que ele melhore na tarefa de fazer previsões.

O processo pode ser resumido em dois grandes loops:

**O Ciclo da Época**
A primeira linha, *for epoch in range(2000):*, inicia o ciclo da época. Uma época é uma passagem completa por todo o conjunto de dados de treinamento. Repetir isso 2000 vezes permite que o modelo refine seus pesos continuamente. A cada nova época, as variáveis running_loss e running_accuracy são zeradas para registrar o desempenho daquela rodada.


**O Ciclo do Mini-Lote**
O loop interno, for data in *train_loader:*, é o coração do aprendizado. Ele processa os dados em pequenos grupos, chamados mini-lotes. Essa é uma abordagem mais eficiente e estável do que processar todos os dados de uma vez.

Dentro desse loop, ocorrem os passos essenciais do treinamento:

Reiniciar os Gradientes: **optimizer.zero_grad()** zera os gradientes acumulados do lote anterior. Pense nisso como "limpar o quadro-negro" antes de começar uma nova lição.

Propagação Direta **(Forward Pass)**: *outputs = classificador(inputs)* executa o modelo. Ele recebe os dados de entrada (inputs) e produz uma previsão (outputs).

Cálculo da Perda e** Retropropagação (Backward Pass)**: *loss = criterion(outputs, labels)* mede o "erro" da previsão comparando-a com o rótulo verdadeiro (labels). Em seguida,*** loss.backward()*** usa a retropropagação para calcular como cada peso do modelo contribuiu para esse erro.

**Atualização dos Pesos: optimizer.step()** usa os gradientes calculados para ajustar os pesos do modelo, dando um pequeno passo na direção certa para reduzir a perda.

**As linhas restantes (ps = F.softmax(outputs), topk, equals, e o print)** servem para monitorar o progresso do treinamento, calculando a acurácia e a perda de cada época e exibindo os resultados para que possamos acompanhar se o modelo está melhorando.

In [9]:
for epoch in range(2000):

    running_loss = 0.
    running_accuracy = 0.

    for data in train_loader:
        inputs, labels = data

        optimizer.zero_grad()

        outputs = classificador(inputs)

        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        ps = F.softmax(outputs)

        top_p, top_class = ps.topk(k = 1, dim = 1)

        equals = top_class == labels.view(*top_class.shape)

        running_accuracy += torch.mean(equals.type(torch.float))

    print('Época {:3d}: perda {:3.5f} - precisão {:3.5f}'.format(epoch + 1, running_loss/len(train_loader), running_accuracy/len(train_loader)))

/tmp/ipykernel_11048/186400110.py:20: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  ps = F.softmax(outputs)


Época   1: perda 1.17219 - precisão 0.30000
Época   2: perda 1.14213 - precisão 0.20000
Época   3: perda 1.11941 - precisão 0.12667
Época   4: perda 1.10595 - precisão 0.18000
Época   5: perda 1.09805 - precisão 0.30000
Época   6: perda 1.07975 - precisão 0.38667
Época   7: perda 1.08539 - precisão 0.34667
Época   8: perda 1.06212 - precisão 0.35333
Época   9: perda 1.05760 - precisão 0.30667
Época  10: perda 1.03887 - precisão 0.32000
Época  11: perda 1.03915 - precisão 0.32667
Época  12: perda 1.02990 - precisão 0.34000
Época  13: perda 1.00834 - precisão 0.33333
Época  14: perda 1.00597 - precisão 0.35333
Época  15: perda 0.99975 - precisão 0.40667
Época  16: perda 0.98392 - precisão 0.40667
Época  17: perda 0.96977 - precisão 0.47333
Época  18: perda 0.95798 - precisão 0.53333
Época  19: perda 0.93355 - precisão 0.62000
Época  20: perda 0.93299 - precisão 0.60667
Época  21: perda 0.91486 - precisão 0.63333
Época  22: perda 0.91392 - precisão 0.61333
Época  23: perda 0.87856 - preci

## Etapa 5: Salvar e carregar o classificador e classificar um registro

In [10]:
torch.save(classificador.state_dict(), 'checkpoint_exer2_computer.pth')

In [11]:
state_dict = torch.load('checkpoint_exer2_computer.pth')
classificador.load_state_dict(state_dict)

<All keys matched successfully>

** 1)** Preparando os Dados e o Modelo
**negrito**

novo = torch.tensor([[3.2, 4.5, 0.9, 1.1]], requires_grad = False)
classificador.eval() *texto em itálico*

novo = ...: Cria um novo tensor que representa uma única amostra de dados. Os valores [3.2, 4.5, 0.9, 1.1] são as **características (ou features**) dessa amostra. O argumento requires_grad = False é importante porque, *como não estamos treinando, não precisamos calcular gradientes*, o que economiza tempo e memória.

**classificador.eval()**: Coloca o modelo (classificador) no modo de avaliação. Isso é essencial porque desativa camadas que se comportam de forma diferente durante o treinamento (como Dropout e BatchNorm), garantindo que o modelo produza resultados consistentes e confiáveis.

**2. Executando a Previsão**

previsao = classificador(novo)
previsao = F.softmax(previsao)
previsao = (previsao > 0.5).numpy() *texto em itálico*

**previsao = classificador(novo):** A amostra novo é passada para o modelo treinado. O modelo faz uma passagem direta (forward pass) e retorna os valores brutos de saída para cada classe.

**previsao = F.softmax(previsao): ** A função softmax converte os valores brutos de saída em probabilidades, garantindo que a soma de todas as probabilidades seja 1. Por exemplo, a saída pode se tornar [0.1, 0.9, 0.0].

**previsao = (previsao > 0.5).numpy():** Esta linha converte as probabilidades em um array binário (True ou False). O valor True é atribuído a qualquer classe com probabilidade maior que 50% (0.5), enquanto o restante é False. O resultado seria algo como [[False, True, False]].



**3. Convertendo a Previsão em Texto**


if previsao[0][0] == True ...
    print('Iris setosa')
elif previsao[0][0] == False and previsao[0][1] == True ...
    print('Iris virginica')
elif previsao[0][0] == False and previsao[0][1] == False and previsao[0][2] == True:
    print('Iris versicolor') *texto em itálico*
    
Este bloco de código if/elif analisa o array binário da previsão para determinar qual classe tem o valor True. Com base nisso, ele imprime o nome da espécie de íris correspondente. Por exemplo, se o array for [[False, True, False]], o segundo elif será ativado, e a mensagem 'Iris virginica' será exibida.


In [12]:
novo = torch.tensor([[3.2, 4.5, 0.9, 1.1]], requires_grad = False)
classificador.eval()
previsao = classificador(novo)
previsao = F.softmax(previsao)
previsao = (previsao > 0.5).numpy()
if previsao[0][0] == True and previsao[0][1] == False and previsao[0][2] == False:
    print('Iris setosa')
elif previsao[0][0] == False and previsao[0][1] == True and previsao[0][2] == False:
    print('Iris virginica')
elif previsao[0][0] == False and previsao[0][1] == False and previsao[0][2] == True:
    print('Iris versicolor')

Iris setosa


/tmp/ipykernel_11048/2632930173.py:4: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  previsao = F.softmax(previsao)
